In [ ]:
"""
======================================================================================
ДОМАШНЕЕ ЗАДАНИЕ 1: ПАРСИНГ КОММЕНТАРИЕВ ПО КРИПТОВАЛЮТЕ BITCOIN
======================================================================================

ЛОГИКА ПАРСИНГА:
---------------
!!! В GOOGLE COLAB НЕ РАБОТАЕТ, РАБОТАЕТ ЧЕРЕЗ YUPYTER В ANACONDA !!!
---------------
1. Источник данных: https://www.investing.com/crypto/bitcoin/chat/
   - Это страница на финансовом портале Investing.com с обсуждениями Bitcoin, где пользователи оставляют комментарии
   - Комментарии организованы постранично (каждая страница имеет номер)

2. Период сбора: 1 год (с 25.10.2024 по 27.10.2025)
   - Дневная гранулярность моделирования

3. Принцип парсинга:
   - Последовательный обход страниц от новых к старым комментариям
   - Парсятся ВСЕ комментарии без фильтрации
   
4. Временные метки:
   - Если комментарий написан только что, сайт показывает "Just now"
   - Если комментарий написан недавно, сайт показывает "X minutes/hours ago"
   - Для старых комментариев указывается точная дата "Oct 26, 2025, 23:33"
   - Функция parse_comment_date() конвертирует все три формата в datetime

5. Дополнительные атрибуты:
   - com_likes: количество лайков (позитивная реакция сообщества)
   - com_dislikes: количество дизлайков (негативная реакция)
   - Эти метрики помогут взвесить важность комментариев при построении индекса сентимента

6. Стратегия работы с перебоями:
   - Парсинг годового периода занимает много времени (~5000 страниц)
   - Возможны ошибки: потеря соединения, блокировка IP, капча, сбои сервера
   - РЕШЕНИЕ: Сохранение промежуточных результатов каждые 500 страниц и всех спарщенных комментариев при сбое (также иногда возникают 
     ошибки в самом начале парсинга - можно просто перезапустить парсер (в крайнем случае несколько раз))
   - После сбоя: вручную меняем стартовую страницу в URL и в переменной current_page в цикле парсингна, перезапускаем
   - В результате получается несколько CSV файлов (например, у меня они назывались: bitcoin_comments_1_436.csv, 
     bitcoin_comments_436_872.csv и т.д.)
   - Финальный шаг: объединение всех файлов с удалением дубликатов (см. функцию ниже)

7. Борьба с защитой сайта:
   - Используется эмуляция реального браузера (Chrome)
   - Скрываются признаки автоматизации (webdriver флаг)
   - Добавлены задержки между действиями
   - Отключена загрузка изображений для ускорения работы
======================================================================================
"""

In [ ]:
!pip install selenium

In [ ]:
# ======================================================================================
# БЛОК 1: ИМПОРТ БИБЛИОТЕК
# ======================================================================================
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime, timedelta
import time
import pandas as pd
import re

In [ ]:
# ======================================================================================
# БЛОК 2: ФУНКЦИЯ ПАРСИНГА ДАТ
# ======================================================================================
def parse_comment_date(date_string):
    """
    Конвертирует различные форматы времени комментариев в единый формат datetime
    
    ПРОБЛЕМА: Сайт investing.com показывает время комментариев в трёх форматах:
    - Появились только что: "Just now"
    - Недавние: "30 minutes ago", "2 hours ago" 
    - Старые: "Oct 26, 2025, 11:56"
    
    РЕШЕНИЕ: Функция распознает формат и конвертирует в datetime для единообразия
    
    Возвращает: datetime объект с точной датой и временем комментария
    """
    now = datetime.now()
    date_string = date_string.strip().lower()
    
    # Паттерн для относительного времени "X minutes/hours ago"
    ago_pattern = r'(\d+)\s+(minute|hour)s?\s+ago'
    match = re.match(ago_pattern, date_string)
    
    if match:
        value = int(match.group(1))
        unit = match.group(2)
        
        if unit == 'minute':
            return now - timedelta(minutes=value)
        elif unit == 'hour':
            return now - timedelta(hours=value)
    
    # Паттерн для абсолютной даты "Oct 26, 2025, 11:56"
    try:
        return datetime.strptime(date_string, "%b %d, %Y, %H:%M")
    except ValueError:
        pass
    
    # Если формат не распознан или "Just now", возвращаем текущее время с предупреждением
    print(f"⚠️ Не удалось распарсить дату: {date_string}")
    return now

In [ ]:
# ======================================================================================
# БЛОК 3: УСТАНОВКА ВРЕМЕННЫХ ГРАНИЦ ПАРСИНГА
# ======================================================================================
# Парсим комментарии за 1 год
START_DATE = datetime(2024, 10, 25)  # Начало периода
END_DATE = datetime(2025, 10, 27)     # Конец периода

In [ ]:
# ======================================================================================
# БЛОК 4: НАСТРОЙКА SELENIUM WEBDRIVER
# ======================================================================================
# ЦЕЛЬ: Замаскировать бота под реального пользователя

options = webdriver.ChromeOptions()

# Headless режим - браузер работает без GUI (быстрее и меньше нагрузка)
options.add_argument('--headless=new')
options.add_argument('--disable-gpu')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# КЛЮЧЕВОЙ МОМЕНТ: Скрываем признаки автоматизации
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('--window-size=1920,1080')

# Используем User-Agent реального браузера Chrome
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

# Отключаем загрузку изображений (ускорение парсинга в 2-3 раза)
prefs = {
    'profile.managed_default_content_settings.images': 2,
    'profile.default_content_setting_values.notifications': 2
}
options.add_experimental_option('prefs', prefs)
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

driver = webdriver.Chrome(options=options)

# Дополнительная маскировка через CDP (Chrome DevTools Protocol)
driver.execute_cdp_cmd('Network.setUserAgentOverride', {
    "userAgent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})
# Скрываем флаг navigator.webdriver (главный признак бота)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")


# ======================================================================================
# БЛОК 5: ЗАГРУЗКА СТРАНИЦЫ И ЗАКРЫТИЕ POP-UP ОКОН
# ======================================================================================
print("🌐 Загружаем страницу...")
driver.get('https://www.investing.com/crypto/bitcoin/chat/')

# Даем время на полную загрузку страницы
time.sleep(5)

# ПРОБЛЕМА: Сайт показывает pop-up окна (согласие с cookies, регистрация)
# РЕШЕНИЕ: Автоматически закрываем их, если появляются

# Закрываем cookie consent banner
try:
    accept_button = WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
    )
    accept_button.click()
    time.sleep(0.5)
except:
    pass  # Если не появился - ничего страшного

# Закрываем окно регистрации
try:
    close_button = WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "svg[data-test='sign-up-dialog-close-button'], svg.float-end"))
    )
    close_button.click()
    time.sleep(0.5)
except:
    pass


# ======================================================================================
# БЛОК 6: ОСНОВНОЙ ЦИКЛ ПАРСИНГА
# ======================================================================================
comments_list = []           # Список для накопления всех комментариев
max_next = 5000              # Максимум страниц (защита от бесконечного цикла)
stop_parsing = False         # Флаг остановки при достижении START_DATE

print("🚀 Начинаем парсинг...")

for i in range(max_next):
    current_page = 1 + i  # Номер текущей страницы (1 - стартовая)
    
    # ОЖИДАНИЕ ЗАГРУЗКИ: Selenium ждет появления комментариев на странице
    try:
        comments = WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.px-1.pb-5.pt-4.transition-colors.duration-300"))
        )
    except:
        print(f"⚠️ Не удалось загрузить комментарии на странице {i+1}")
        continue
    
    # ПАРСИНГ КОММЕНТАРИЕВ НА ТЕКУЩЕЙ СТРАНИЦЕ
    if len(comments) > 0:
        for comment in comments:
            try:
                # Извлекаем текст комментария
                com_text = comment.find_element(By.CSS_SELECTOR, "div.break-words.leading-5").text
                if not com_text or com_text.strip() == "":
                    continue  # Пропускаем пустые комментарии
                
                # Извлекаем и парсим дату
                com_date_str = comment.find_element(By.CSS_SELECTOR, "span[data-test='comment-date']").text
                com_date = parse_comment_date(com_date_str)
                
                # ПРОВЕРКА ВРЕМЕННЫХ ГРАНИЦ
                # Если достигли начальной даты - останавливаем парсинг
                if com_date < START_DATE:
                    print(f"⛔ Достигнута начальная дата: {com_date_str}")
                    stop_parsing = True
                    break
                
                # Пропускаем комментарии новее конечной даты (на случай будущих дат)
                if com_date > END_DATE:
                    continue
                
                # Извлекаем лайки и дизлайки
                buttons = comment.find_elements(By.CSS_SELECTOR, "button.group.flex")
                com_likes = buttons[0].text if len(buttons) > 0 else "0"
                com_dislikes = buttons[1].text if len(buttons) > 1 else "0"
                
                # Добавляем комментарий в список
                comments_list.append({
                    "com_text": com_text,
                    "com_likes": com_likes,
                    "com_dislikes": com_dislikes,
                    "com_date": com_date.strftime("%Y-%m-%d %H:%M:%S"),
                    "com_date_original": com_date_str
                })
            except:
                # Пропускаем комментарии с ошибками парсинга
                continue
    else:
        print(f"⚠️ Страница {i+1}: комментарии не найдены - пропускаем")
    
    # ВЫВОД ПРОГРЕССА: Каждые 50 страниц показываем статистику
    if (i + 1) % 50 == 0:
        print(f"📄 Страница {i+1}. Всего собрано комментариев: {len(comments_list)}")
    
    # ОСТАНОВКА при достижении начальной даты
    if stop_parsing:
        print(f"✅ Парсинг завершен на странице {i+1}")
        break
    
    # КРИТИЧЕСКИ ВАЖНО: Промежуточное сохранение каждые 500 страниц
    # Это защищает от потери данных при сбоях
    if (i + 1) % 500 == 0:
        pd.DataFrame(comments_list).to_csv(
            f'bitcoin_comments_{i-499}_{i+1}.csv',  # Имя показывает диапазон страниц
            sep='\t',
            index=False,
            encoding='utf-8'
        )
        print(f"💾 Backup сохранен: страницы {i-499}-{i+1}, комментариев: {len(comments_list)}")
    
    # ПЕРЕХОД НА СЛЕДУЮЩУЮ СТРАНИЦУ
    try:
        next_page_num = current_page + 1
        next_button = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.XPATH, f"//button[contains(@class,'text-link') and text()='{next_page_num}']"))
        )
        # Используем JavaScript клик (надежнее обычного клика)
        driver.execute_script("arguments[0].click();", next_button)
    except:
        print(f"⚠️ Кнопка Next не найдена на странице {i+1}")
        break

# ======================================================================================
# БЛОК 7: ФИНАЛЬНОЕ СОХРАНЕНИЕ
# ======================================================================================
df = pd.DataFrame(comments_list)
df.to_csv('bitcoin_comments_final.csv', sep='\t', index=False, encoding='utf-8')
print(f"\n✅ Парсинг завершен! Сохранено {len(comments_list)} комментариев из {i+1} страниц")
driver.quit()

In [ ]:
# ======================================================================================
# БЛОК 8: ОБЪЕДИНЕНИЕ НЕСКОЛЬКИХ CSV ФАЙЛОВ (после перезапусков)
# ======================================================================================
"""
ПРОБЛЕМА ДОЛГОГО ПАРСИНГА:
- При парсинге 1000+ страниц возможны сбои: потеря соединения, блокировка IP, капча, зависание браузера
- При сбое часть данных теряется

РЕШЕНИЕ - СТРАТЕГИЯ CHECKPOINT'ОВ:
1. Каждые 500 страниц создается backup файл (см. выше)
2. При сбое: смотрим последний сохраненный файл
3. Меняем стартовую страницу в URL: driver.get('https://...chat/[НОМЕР_СТРАНИЦЫ]')
4. Перезапускаем парсинг с этой страницы
5. Получаем несколько файлов: bitcoin_comments_1_500.csv, bitcoin_comments_500_1000.csv и т.д.
6. Финальный шаг: объединяем все файлы этой функцией
"""

import glob

def merge_comment_files(input_pattern, output_file):
    """
    Объединяет несколько CSV файлов с комментариями в один Excel файл.
    Автоматически удаляет дубликаты (возникают на границах файлов).
    
    Args:
        input_pattern: паттерн поиска, например "bitcoin_comments_*.csv"
        output_file: имя итогового файла, например "bitcoin_comments_merged.xlsx"
    """
    
    # Находим все файлы с комментариями
    csv_files = glob.glob(input_pattern)
    
    if not csv_files:
        print(f"❌ Файлы не найдены по паттерну: {input_pattern}")
        return
    
    print(f"📂 Найдено файлов: {len(csv_files)}")
    print(f"   Список файлов: {csv_files}")
    
    all_dataframes = []
    
    # Читаем каждый CSV файл
    for file in csv_files:
        print(f"\n📖 Читаю файл: {file}")
        try:
            # Используем разделитель табуляция (как при сохранении)
            df = pd.read_csv(file, sep='\t', encoding='utf-8')
            all_dataframes.append(df)
            print(f"   ✓ Загружено строк: {len(df)}")
        except Exception as e:
            print(f"   ✗ Ошибка при чтении файла {file}: {e}")
    
    if not all_dataframes:
        print("❌ Не удалось загрузить ни один файл")
        return
    
    # Объединяем все данные в один датафрейм
    merged_df = pd.concat(all_dataframes, ignore_index=True)
    print(f"\n📊 Всего строк до удаления дубликатов: {len(merged_df)}")
    
    # УДАЛЕНИЕ ДУБЛИКАТОВ
    # Дубликаты возникают потому что:
    # 1. При перезапуске парсим несколько страниц назад для надежности
    # 2. Некоторые комментарии могли быть в нескольких backup'ах
    # Критерий дубликата: одинаковый текст и дата комментария
    merged_df = merged_df.drop_duplicates(
        subset=['com_text', 'com_date'], 
        keep='first'  # Оставляем первое вхождение
    )
    print(f"📊 Строк после удаления дубликатов: {len(merged_df)}")
    
    # Сортируем по дате (от новых к старым)
    if 'com_date' in merged_df.columns:
        merged_df = merged_df.sort_values('com_date', ascending=False)
    
    # Сбрасываем индексы после удаления дубликатов
    merged_df = merged_df.reset_index(drop=True)
    
    # ФИНАЛЬНОЕ СОХРАНЕНИЕ В EXCEL
    merged_df.to_excel(output_file, index=False, engine='openpyxl')
    print(f"\n✅ Файл успешно сохранён: {output_file}")
    print(f"📈 Итого уникальных комментариев: {len(merged_df)}")
    print(f"📅 Период: с {merged_df['com_date'].min()} по {merged_df['com_date'].max()}")
    
    return merged_df


# ЗАПУСК ОБЪЕДИНЕНИЯ
if __name__ == "__main__":
    # Объединяем все файлы bitcoin_comments_*.csv в один итоговый Excel
    merge_comment_files("bitcoin_comments_*.csv", "bitcoin_comments_merged.xlsx")